## ⚡ Zero-Shot Forecasting with EnergyTTM

This notebook performs **zero-shot energy forecasting** using a pretrained EnergyTTM model without task-specific fine-tuning.

The workflow includes:
- Loading pretrained model weights
- Preparing time-series energy data
- Running forward inference for forecasting
- Evaluating predictions using standard metrics

The objective is to assess the generalization capability of EnergyTTM on unseen buildings in a zero-shot setting.

In [ ]:
import os
import random 
import math 
import tempfile 
import torch 
import pickle 
import logging 
import warnings
import json
import torch.nn as nn

import matplotlib.pyplot as plt
from tqdm import tqdm
import numpy as np 
import pandas as pd
from transformers import Trainer, TrainingArguments, set_seed, EarlyStoppingCallback
from torch.utils.data import ConcatDataset, Dataset, DataLoader
from tsfm_public.models.tinytimemixer.configuration_tinytimemixer import TinyTimeMixerConfig
from tsfm_public.models.tinytimemixer import TinyTimeMixerForPrediction

warnings.filterwarnings("ignore")
SEED = 42
set_seed(SEED)


### Evaluation Metrics

  Root mean squared error normalized by the mean load and expressed as a percentage.  
  Particularly suitable for daily energy series (24-hour structure assumed).



In [ ]:

def cal_nrmse(pred, true, eps=1e-8):
    true = np.array(true)
    pred = np.array(pred)

    M = len(true) // 24
    y_bar = np.mean(true)
    NRMSE = 100 * (1/ (y_bar+eps)) * np.sqrt((1 / (24 * M)) * np.sum((true - pred) ** 2))
    return NRMSE


In [3]:


def standardize_series(series, eps=1e-8):
    mean = np.mean(series)
    std = np.std(series)
    standardized_series = (series - mean) / (std+eps)
    return standardized_series, mean, std

def unscale_predictions(predictions, mean, std, eps=1e-8):
    return predictions * (std+eps) + mean


class TimeSeriesDataset(Dataset):
    def __init__(self, data, backcast_length, forecast_length, stride=1):
        # Standardize the time series data
        self.data, self.mean, self.std = standardize_series(data)
        self.backcast_length = backcast_length
        self.forecast_length = forecast_length
        self.stride = stride

    def __len__(self):
        return (len(self.data) - self.backcast_length - self.forecast_length) // self.stride + 1

    def __getitem__(self, index):
        start_index = index * self.stride
        x = self.data[start_index : start_index + self.backcast_length]
        y = self.data[start_index + self.backcast_length : start_index + self.backcast_length + self.forecast_length]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)


### 🛠️ Helper Functions for the Evaluation Pipeline

These helper functions modularize the testing workflow to improve readability, reusability, and debugging. They handle building selection, data cleaning, dataset preparation, inference, and metric computation in clearly separated steps. This structured design keeps the main evaluation loop clean and maintainable.

- **Building Selection** – Identify which building columns to evaluate (`"all"`, single, or multiple).
- **Data Cleaning** – Replace missing values using median imputation.
- **Dataset Validation** – Ensure sufficient sequence length and construct sliding-window samples.
- **Inference Execution** – Run model forward passes and collect predictions.
- **Metric Evaluation** – Unscale outputs and compute CVRMSE, NRMSE, and MAE.


In [ ]:
def get_buildings_to_test(df, target_buildings):
    """
    Determine which building IDs (columns) to evaluate.

    Parameters:
        df (DataFrame): Input dataframe where columns correspond to buildings.
        target_buildings (str or list):
            - "all"  -> evaluate all buildings
            - string -> evaluate a single building
            - list   -> evaluate a specific list of buildings

    Returns:
        list: List of building IDs to test.
    """
    if target_buildings == "all":
        return list(df.columns)
    elif isinstance(target_buildings, str):
        return [target_buildings]
    elif isinstance(target_buildings, list):
        return target_buildings
    else:
        raise ValueError("target_buildings must be 'all', a string, or list")



In [ ]:

def clean_series(energy_data):
    """
    Replace NaN values in a time series with the median of the series.

    Parameters:
        energy_data (np.array): Raw energy time series.

    Returns:
        np.array: Cleaned energy time series.
    """
    nan_count = np.isnan(energy_data).sum()
    
    if nan_count > 0:
        # Compute median ignoring NaNs
        median_val = np.nanmedian(energy_data)
        
        # Replace NaNs with median value
        energy_data = np.where(np.isnan(energy_data), median_val, energy_data)
        
        print(f"  Filled {nan_count} NaNs with median={median_val:.4f}")
    
    return energy_data


In [ ]:
def create_dataset_if_valid(energy_data, args):
    """
    Create a TimeSeriesDataset if the series length is sufficient.

    Parameters:
        energy_data (np.array): Cleaned energy time series.
        args (dict): Configuration containing:
                     - context_length
                     - prediction_length
                     - patch_stride

    Returns:
        TimeSeriesDataset or None: Dataset object if valid, otherwise None.
    """
    # Minimum required length to form at least one training sample
    min_required = args["context_length"] + args["prediction_length"]

    # Skip if series is too short
    if len(energy_data) < min_required:
        print("   Too short, skipping...")
        return None

    # Create dataset using sliding windows
    dataset = TimeSeriesDataset(
        energy_data,
        args["context_length"],
        args["prediction_length"],
        args["patch_stride"]
    )

    # Ensure dataset contains at least one sample
    if len(dataset) == 0:
        print("   No samples, skipping...")
        return None

    return dataset



In [ ]:



def run_inference(model, dataset, criterion, device):
    """
    Run model inference on a dataset and collect predictions.

    Parameters:
        model (torch.nn.Module): Trained forecasting model.
        dataset (Dataset): TimeSeriesDataset object.
        criterion (loss function): Loss function (e.g., MSE).
        device (torch.device): CPU or GPU device.

    Returns:
        tuple or None:
            (y_true, y_pred, avg_loss) if valid samples exist,
            otherwise None.
    """
    model.eval()

    val_losses = []
    y_true_test = []
    y_pred_test = []

    # Iterate over dataset one sample at a time
    for x_test, y_test in DataLoader(dataset, batch_size=1):

        # Add channel dimension and move to device
        x_test = x_test.unsqueeze(-1).to(device)
        y_test = y_test.to(device)

        with torch.no_grad():
            # Forward pass
            output = model(x_test)
            forecast = output.prediction_outputs.squeeze(-1)

            # Compute loss
            loss = criterion(forecast, y_test)

            # Skip if loss is NaN
            if torch.isnan(loss):
                continue

            # Store results
            val_losses.append(loss.item())
            y_true_test.append(y_test.cpu().numpy())
            y_pred_test.append(forecast.cpu().numpy())

    # If no valid predictions collected
    if len(y_true_test) == 0:
        return None

    # Concatenate predictions and ground truth
    y_true = np.concatenate(y_true_test, axis=0)
    y_pred = np.concatenate(y_pred_test, axis=0)

    # Return true values, predictions, and average validation loss
    return y_true, y_pred, np.mean(val_losses)



In [ ]:
# -------------------------------------------------------
# Load Model Configuration
# -------------------------------------------------------
config_file = '../Energy-TTM/config/tinyTimeMixers.json'

# Read hyperparameters from JSON config
with open(config_file, 'r') as f:
    args = json.load(f)

# -------------------------------------------------------
# Device Setup (GPU if available, else CPU)
# -------------------------------------------------------
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

# -------------------------------------------------------
# Initialize Model
# -------------------------------------------------------



model = TinyTimeMixerForPrediction.from_pretrained(
    "sriv-naman-iisc/Energy-FM-v1",  # Name of the model on Hugging Face
    revision="energy-ttm",
    num_input_channels=1,  # tsp.num_input_channels 
).to(device)
# -------------------------------------------------------
# Model Statistics
# -------------------------------------------------------
# Count trainable parameters
param = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Model's parameter count is:", param)

# -------------------------------------------------------
# Define Loss Function
# -------------------------------------------------------
# Mean Squared Error for forecasting evaluation
criterion = torch.nn.MSELoss()

# -------------------------------------------------------
# Run Zero-Shot Testing
# -------------------------------------------------------
# Evaluate pretrained model on forecasting dataset
test(
    args=args,
    model=model,
    criterion=criterion,
    dataset_path="../Dataset/Forecasting",
    result_path="test_results_zeroshot",
    device=device
)


Model's parameter count is: 28858

Testing file: Bareilly-1H
   ▶ Building: BR02
  Filled 9984 NaNs with median=0.1660
   CVRMSE=0.0974, NRMSE=236.3778, MAE=0.0428

Testing file: Mathura-1H
 BR02 not found, skipping...

Testing complete!
Results saved at: test_results_zeroshot/test_results.csv
